# GraphGenerator on ZINC

This notebook demonstrates the two-stage `GraphGenerator`: first generate a new interpretation graph, then instantiate base molecules conditionally from nearby ZINC examples.

In [1]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from collections import Counter

from nsppk import NSPPK
from sklearn.ensemble import RandomForestClassifier

from abstractgraph.display import display, display_decomposition_graph, display_mappings
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.operators import *
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.edge_generator import EdgeGenerator
from abstractgraph_generative.graph_generator import GraphGenerator
from abstractgraph_ml.feasibility import FeasibilityEstimator, FeasibilityEstimatorFeatureCannotExist



/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [3]:
def draw(graph, decomposition_function, *, nbits=11, label_mode="operator", size=(12, 6), n_elements_per_row=8):
    ag = graph_to_abstract_graph(
        graph,
        decomposition_function=decomposition_function,
        nbits=nbits,
        label_mode=label_mode,
    )
    display(ag, size=size)
    display_mappings(ag, n_elements_per_row=n_elements_per_row)
    return ag

---

In [4]:
loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 3000
min_num_nodes = 30
max_num_nodes = 50

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)

print(f"dataset: {dataset_name}")
print(f"n_graphs: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")


dataset: zinc_250k
n_graphs: 3000
node_range: [30, 50]


In [5]:
label_mode = "operator" #label_mode: str = "operator" (default) or "histogram" or "histogram_values" for AbstractGraph node labeling.
nbits = 14
cycle_tree = add(compose(name("cycle"), cycle()), compose(name("tree"), tree()))
decomposition_function = compose(intersection_edges(), cycle_tree)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
feasibility_kwargs = dict(
    nbits=19,
    parallel=True,
    backend="loky",
    n_jobs=-1,
)
partial_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
partial_feasibility_estimator = FeasibilityEstimator(partial_feasibility_estimators)

final_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
final_feasibility_estimator = FeasibilityEstimator(final_feasibility_estimators)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
edge_estimator_vectorizer = NSPPK(radius=1, distance=4, connector=0, nbits=14, dense=True, parallel=True)
edge_graph_estimator = GraphEstimator(
    transformer=edge_estimator_vectorizer,
    estimator=RandomForestClassifier(
        n_estimators=80,
        random_state=0,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
)
edge_generator = EdgeGenerator(
    partial_feasibility_estimator=partial_feasibility_estimator,
    final_feasibility_estimator=final_feasibility_estimator,
    graph_estimator=edge_graph_estimator,
    n_negative_per_positive=3,
    n_replicates=2,
    beam_size=3,
    max_restarts=2,
    fit_n_jobs=-1,
    fit_backend="loky",
    seed=0,
)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
context_vectorizer = NSPPK(radius=1, distance=4, connector=0, nbits=14, dense=True, parallel=True)
conditional_generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    base_cut_radius=0,
    interpretation_cut_radius=1,
    context_vectorizer=context_vectorizer,
    n_jobs=1,
)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
generator = GraphGenerator(
    edge_generator=edge_generator,
    conditional_generator=conditional_generator,
    seed=None,
    debug=True,
    require_new_interpretation_graph=True,
    max_same_interpretation_retries=3,
)

In [6]:
%%time
generator.store(graphs)

CPU times: user 4min 15s, sys: 5.79 s, total: 4min 21s
Wall time: 4min 16s


In [ ]:
%%time
n_samples = 4
n_instances_per_sample = 3
generated_graphs = generator.sample(
    n_samples=n_samples,
    n_interpretation_neighbors=100,
    n_conditional_neighbors=300,
    n_instances_per_sample=n_instances_per_sample,
    interpretation_edge_removal_size=1,  # 0 bypasses edge generation; 1 removes all interpretation edges before regrowth.
    random_state=None,
    conditional_generate_kwargs=dict(
        random_state=None,
        max_backtracks=2000,
        max_attempts_per_sample=6,
        require_signature_coverage=True,
    ),
)

generated_groups = [levels[0] for levels in generator.last_level_generated_graphs_history_]
generated_interpretation_graphs = [levels[-1][0] for levels in generator.last_level_generated_graphs_history_]

print(f"generated molecules: {len(generated_graphs)}")
print("attempted seed indices:", generator.last_sampled_indices_)
print("successful seed indices:", generator.last_successful_sampled_indices_)
print("successful interpretation targets:", len(generated_interpretation_graphs))
if not generated_graphs:
    print("No molecules generated; inspect warnings and try a larger neighborhood or dataset slice.")

[graph-generator sample] event=seed_start currently_generated=0/4 attempted_seeds=1/40 generated_graphs=0 seed_idx=1012
[graph-generator edge] seed_idx=1012 n_neighbors=100 neighbor_indices=[866, 1626, 1356, 1621, 307, 530, 140, 766, 1619, 1069, 814, 1424, 110, 330, 322, 1178, 1392, 580, 77, 1174, 1771, 2940, 106, 344, 562, 198, 276, 2347, 225, 2640, 1517, 2796, 614, 100, 868, 19, 811, 1390, 26, 475, 2173, 168, 504, 401, 2196, 210, 763, 775, 2384, 1537, 1858, 1904, 57, 365, 619, 1985, 250, 1307, 688, 978, 87, 2141, 78, 648, 273, 660, 2307, 2832, 1350, 1837, 238, 2168, 616, 1613, 1700, 1196, 285, 306, 103, 112, 171, 606, 1029, 904, 1082, 573, 714, 2255, 2969, 1421, 798, 2352, 778, 2189, 1510, 1140, 574, 687, 2410, 1979] neighbor_distances=[16.8226, 18.5203, 18.868, 19.3907, 19.9249, 20.0, 20.0499, 20.199, 20.2237, 20.445, 20.6882, 21.1424, 21.2603, 21.4476, 21.7025, 21.9089, 22.2711, 22.3159, 22.5389, 22.6274, 22.7596, 22.7596, 22.8035, 22.8254, 22.891, 23.0217, 23.0217, 23.0217, 23.108

/Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/abstractgraph-ecosystem/repos/abstractgraph-generative/src/abstractgraph_generative/conditional.py:2830: RuntimeWarning: generate requested n_samples=3 but produced 0 after 72 attempts (budget=72).
  warnings.warn(


[DEBUG] event=generate_progress
[DEBUG]   attempts_per_sec=2.64
[DEBUG]   completed_futures=27
[DEBUG]   constructed=1
[DEBUG]   constructed_rate=0.037
[DEBUG]   eta_seconds=20.5
[DEBUG]   filtered=0
[DEBUG]   filtered_fraction=0.000
[DEBUG]   keep_rate=0.037
[DEBUG]   kept=1
[DEBUG]   phase=sequential
[DEBUG]   remaining_samples=2
[DEBUG]   submitted_attempts=27
[DEBUG] event=generate_progress
[DEBUG]   attempts_per_sec=2.71
[DEBUG]   completed_futures=55
[DEBUG]   constructed=1
[DEBUG]   constructed_rate=0.018
[DEBUG]   eta_seconds=40.6
[DEBUG]   filtered=0
[DEBUG]   filtered_fraction=0.000
[DEBUG]   keep_rate=0.018
[DEBUG]   kept=1
[DEBUG]   phase=sequential
[DEBUG]   remaining_samples=2
[DEBUG]   submitted_attempts=55
[DEBUG] event=generate_progress
[DEBUG]   attempts_per_sec=2.70
[DEBUG]   completed_futures=72
[DEBUG]   constructed=2
[DEBUG]   constructed_rate=0.028
[DEBUG]   eta_seconds=13.3
[DEBUG]   filtered=0
[DEBUG]   filtered_fraction=0.000
[DEBUG]   keep_rate=0.028
[DEBUG] 

/Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/abstractgraph-ecosystem/repos/abstractgraph-generative/src/abstractgraph_generative/conditional.py:2830: RuntimeWarning: generate requested n_samples=3 but produced 2 after 72 attempts (budget=72).
  warnings.warn(


[graph-generator edge] seed_idx=1347 n_neighbors=100 neighbor_indices=[2489, 1140, 103, 162, 1527, 381, 1619, 280, 926, 1979, 307, 752, 660, 196, 1237, 1392, 1669, 1537, 694, 847, 1196, 18, 1550, 614, 42, 605, 1597, 242, 2274, 494, 862, 447, 616, 147, 619, 26, 1204, 171, 688, 499, 250, 606, 78, 120, 677, 155, 574, 81, 350, 1642, 100, 112, 1082, 1626, 582, 238, 720, 144, 87, 2546, 2944, 1458, 2740, 858, 401, 322, 814, 2321, 1421, 5, 449, 766, 92, 2196, 516, 1356, 1178, 1518, 2384, 2189, 1015, 58, 2358, 2141, 2241, 1621, 306, 778, 1904, 517, 1022, 225, 330, 21, 416, 573, 1031, 686, 1073, 2671] neighbor_distances=[0.0, 13.7477, 14.0357, 14.0357, 17.3781, 17.7764, 18.4662, 18.6011, 18.7617, 18.7617, 19.1572, 19.3132, 19.6214, 19.6723, 19.9249, 20.0, 20.1246, 20.2237, 20.445, 20.445, 20.5183, 20.5913, 20.6882, 20.7364, 20.9762, 21.1896, 21.1896, 21.2603, 21.6102, 21.7486, 21.7486, 21.7715, 21.8403, 22.0, 22.2486, 22.383, 22.383, 22.4722, 22.5832, 22.6274, 22.8254, 22.8254, 22.9129, 22.9129,

In [ ]:
if not generated_graphs:
    print("No generated molecules to display.")
else:
    for sample_idx, (seed_idx, seed_levels, generated_levels) in enumerate(zip(generator.last_successful_sampled_indices_, generator.last_level_seed_graphs_history_, generator.last_level_generated_graphs_history_)):
        seed_graph = seed_levels[0]
        generated_instances = generated_levels[0]
        print("seed molecule")
        display_graphs([seed_graph], n_graphs_per_line=1)
        
        print(f"generated molecule instances ({len(generated_instances)})")
        display_graphs(generated_instances, n_graphs_per_line=n_instances_per_sample)

In [ ]:
if not generated_graphs:
    print("No generated molecules to display.")
else:
    for sample_idx, (seed_idx, seed_levels, generated_levels) in enumerate(zip(generator.last_successful_sampled_indices_, generator.last_level_seed_graphs_history_, generator.last_level_generated_graphs_history_)):
        seed_graph = seed_levels[0]
        seed_interpretation_graph = seed_levels[-1]
        generated_interpretation_graph = generated_levels[-1][0]
        generated_instances = generated_levels[0]
        print("=" * 120)
        print(f"sample {sample_idx} | seed index {seed_idx}")

        print("seed molecule")
        display_graphs([seed_graph], n_graphs_per_line=1)

        print("seed interpretation graph")
        display([seed_interpretation_graph], size=(5, 4))

        print("generated interpretation graph")
        display([generated_interpretation_graph], size=(5, 4))

        print(f"generated molecule instances ({len(generated_instances)})")
        display_graphs(generated_instances, n_graphs_per_line=n_instances_per_sample)
        for generated_instance in generated_instances:
            draw(generated_instance, decomposition_function=decomposition_function, nbits=nbits, label_mode=label_mode)
